In [1]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime
from skopt.space import Real
from skopt import gp_minimize
from scipy.optimize import minimize

In [2]:
#Constants for dates

start_dl = "2023-01-01"
end_dl = "2026-11-26"

In [3]:
# INITIAL PORTFOLIO FILTER
# 1. Must be a company stock listed in the US or Canada
# 2. Must trade in USD or CAD
# 3. Average volume must be at least 5000
# 4. Exclude months with fewer than 18 trading days

# Read the tickers
try:
    tickers = pd.read_csv("Tickers_Example.csv", header=None)
    tickers = tickers.dropna()
except Exception as e:
    raise ValueError("Could not read ticker file") from e

eligible = []
eligible_currency = []
checked = []  # NEW: track tickers already checked
stock_info = {}  # NEW: save stock information for later

for i in range(len(tickers)):
    # NEW: remove spaces and make tickers uppercase
    symbol = str(tickers.iloc[i, 0]).strip().upper()

    # NEW: skip empty or duplicate tickers
    if symbol == "" or symbol in checked:
        continue

    checked.append(symbol)
    valid = True

    try:
        stock = yf.Ticker(symbol)
        info = stock.info

        # Company stock?
        if info.get("quoteType") != "EQUITY":
            valid = False

        # Listed in US/Canada?
        if info.get("market") not in ["us_market", "ca_market"]:
            valid = False

        # Currency in USD/CAD?
        if info.get("currency") not in ["USD", "CAD"]:
            valid = False

        # Check average trading volume
        if valid:
            hist = stock.history(
                start=start_dl,
                end=end_dl
            )

            # NEW: check if volume data is missing
            if hist.empty or "Volume" not in hist.columns:
                valid = False
            else:
                dates = pd.to_datetime(hist.index)
                volume = hist["Volume"].replace(
                    [np.inf, -np.inf], np.nan
                )

                # NEW: remove invalid volume values
                volume = volume.where(volume >= 0)

                days_per_month = volume.groupby(
                    [dates.year, dates.month]
                ).count()

                volume_per_month = volume.groupby(
                    [dates.year, dates.month]
                ).sum()

                total_days = 0
                total_volume = 0

                for m in range(len(days_per_month)):
                    if days_per_month.iloc[m] >= 18:
                        total_days += days_per_month.iloc[m]
                        total_volume += volume_per_month.iloc[m]

                if total_days == 0:
                    valid = False
                elif total_volume / total_days < 5000:
                    valid = False

    # NEW: handle failed ticker requests
    except Exception as e:
        print(symbol, "could not be checked:", e)
        valid = False

    if valid:
        eligible.append(symbol)
        eligible_currency.append(info.get("currency"))
        stock_info[symbol] = info


# Download historical prices
stock_data = {}


if len(eligible) > 0:
    try:
        all_stock = yf.download(
            eligible,
            start=start_dl,
            end=end_dl,
            group_by="ticker",
            auto_adjust=False,
            progress=False
        )

        for ticker in eligible:
            try:
                # NEW: handle single and multiple stock downloads
                if isinstance(all_stock.columns, pd.MultiIndex):
                    if ticker in all_stock.columns.get_level_values(0):
                        indiv_stock = all_stock[ticker].copy()
                    else:
                        indiv_stock = all_stock.xs(
                            ticker, axis=1, level=1
                        ).copy()
                else:
                    indiv_stock = all_stock.copy()

                required = ["High", "Low", "Close"]

                # NEW: check if any price columns are missing
                if not all(col in indiv_stock.columns for col in required):
                    print(ticker, "has missing price columns")
                    continue

                # NEW: remove rows with missing or invalid prices
                indiv_stock = indiv_stock.dropna(subset=required)

                for column in required:
                    indiv_stock = indiv_stock[
                        np.isfinite(indiv_stock[column]) &
                        (indiv_stock[column] > 0)
                    ]

                # NEW: check if enough historical data is available
                if len(indiv_stock) < 252:
                    print(ticker, "does not have enough price data")
                    continue

                # NEW: check if the stock has recent prices
                last_date = pd.Timestamp(indiv_stock.index[-1])
                last_date = last_date.tz_localize(None)

                check_date = min(
                    pd.Timestamp.today().normalize(),
                    pd.Timestamp(end_dl)
                )

                if (check_date - last_date).days > 7:
                    print(ticker, "has outdated price data")
                    continue

                indiv_stock.index = indiv_stock.index.strftime("%Y-%m-%d")
                stock_data[ticker] = indiv_stock

            # NEW: skip stocks with failed downloads
            except (KeyError, TypeError, ValueError) as e:
                print(ticker, "could not be downloaded:", e)

    # NEW: handle failed batch downloads
    except Exception as e:
        print("Historical download failed:", e)

# Convert USD prices to CAD
cadusd = pd.Series(dtype=float)

if len(stock_data) > 0:
    try:
        cadusd = yf.Ticker("CADUSD=X").history(
            start=start_dl,
            end=end_dl
        )["Close"]

        cadusd.index = cadusd.index.strftime("%Y-%m-%d")

        # NEW: remove invalid exchange rates
        cadusd = cadusd.replace([np.inf, -np.inf, 0], np.nan)
        cadusd = cadusd.dropna()

    except Exception as e:
        print("Exchange rates could not be downloaded:", e)


# Convert the remaining stocks
for ticker in list(stock_data):
    table = stock_data[ticker]
    i = eligible.index(ticker)

    if eligible_currency[i] == "USD":
        # NEW: match exchange rates to stock dates
        dates = cadusd.index.union(table.index)
        rates = cadusd.reindex(dates).sort_index().ffill()
        table["CADUSD"] = rates.reindex(table.index)

        table["Close_CAD"] = table["Close"] / table["CADUSD"]
        table["High_CAD"] = table["High"] / table["CADUSD"]
        table["Low_CAD"] = table["Low"] / table["CADUSD"]

    else:
        table["Close_CAD"] = table["Close"]
        table["High_CAD"] = table["High"]
        table["Low_CAD"] = table["Low"]

    required = ["Close_CAD", "High_CAD", "Low_CAD"]

    # NEW: remove invalid CAD prices
    table[required] = table[required].replace(
        [np.inf, -np.inf], np.nan
    )

    table = table.dropna(subset=required)

    for column in required:
        table = table[table[column] > 0]

    # NEW: check that enough valid CAD prices remain
    if len(table) < 252:
        print(ticker, "does not have enough valid CAD prices")
        del stock_data[ticker]
        continue

    # NEW: check for recent CAD prices
    last_date = pd.Timestamp(table.index[-1])

    if (check_date - last_date).days > 7:
        print(ticker, "has outdated CAD prices")
        del stock_data[ticker]
        continue

    stock_data[ticker] = table


# NEW: update eligible lists after removing stocks
final_eligible = []
final_currency = []

for i in range(len(eligible)):
    if eligible[i] in stock_data:
        final_eligible.append(eligible[i])
        final_currency.append(eligible_currency[i])

eligible = final_eligible
eligible_currency = final_currency

print("Eligible tickers:", eligible)
print("Number of eligible stocks:", len(eligible))

Eligible tickers: ['AAPL', 'ABBV', 'ABT', 'ACN', 'AIG', 'AMZN', 'AXP', 'BA', 'BAC', 'BB.TO', 'BIIB', 'BLK', 'BMY', 'C', 'CAT', 'CL', 'KO', 'LLY', 'LMT', 'MO', 'MRK', 'PEP', 'PFE', 'RY.TO', 'PM', 'PYPL', 'QCOM', 'SHOP.TO', 'T.TO', 'TD.TO', 'TXN', 'UNH', 'UNP', 'UPS', 'USB']
Number of eligible stocks: 35


## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** XX  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [4]:
def pull_market_cap(ticker, ticker_df, target_date):

    close_price = ticker_df.loc[:target_date].iloc[-1]["Close_CAD"]

    target_dt = pd.to_datetime(target_date)
    end_dt = target_dt + pd.Timedelta(days=5)
    shares_history = ticker.get_shares_full(start=target_dt.strftime("%Y-%m-%d"),
                                            end=end_dt.strftime("%Y-%m-%d"))

    if shares_history is not None and len(shares_history) > 0:
        share_count = shares_history.iloc[0]
    else:
        share_count = ticker.info.get("sharesOutstanding")

    if share_count is None:
        return np.nan
    return close_price * share_count


In [5]:
def calculate_adx(data, period=14):

    if len(data) < 2 * period:
        return pd.DataFrame(np.nan, index=data.index, columns=["adx", "DX", "+DI", "-DI", "smoothed TR"])

    data = data[["High_CAD", "Low_CAD", "Close_CAD"]].copy()

    # 1. True Range
    high_low = data["High_CAD"] - data["Low_CAD"]
    high_close = abs(data["High_CAD"] - data["Close_CAD"].shift(1))
    low_close = abs(data["Low_CAD"] - data["Close_CAD"].shift(1))

    data["Trading Range"] = high_low
    data["up gap"] = high_close
    data["down gap"] = low_close

    data["True Range"] = pd.concat(
        [high_low, high_close, low_close], axis=1
    ).max(axis=1)

    # 2. Directional Momentum
    up_move = data["High_CAD"].diff()
    down_move = -data["Low_CAD"].diff()

    pos_cond = (up_move > down_move) & (up_move > 0)
    neg_cond = (down_move > up_move) & (down_move > 0)

    data["+DM"] = np.where(pos_cond, up_move, 0)
    data["-DM"] = np.where(neg_cond, down_move, 0)

    # 3. Wilder Smoothing
    def wilder_smoothing(series, smoothing_period):
        smoothed = {}
        initial = sum(series.values[1:smoothing_period + 1])
        smoothed[smoothing_period] = initial

        for i in range(smoothing_period + 1, len(series)):
            smoothed[i] = (
                smoothed[i - 1]
                - (smoothed[i - 1] / smoothing_period)
                + series.values[i])

        return pd.Series(list(smoothed.values()), index=series.index[smoothing_period:])

    data["smoothed +DM"] = wilder_smoothing(data["+DM"], period)
    data["smoothed -DM"] = wilder_smoothing(data["-DM"], period)
    data["smoothed TR"] = wilder_smoothing(data["True Range"], period)

    # 4. Directional Index
    data["+DI"] = (data["smoothed +DM"] / data["smoothed TR"]) * 100
    data["-DI"] = (data["smoothed -DM"] / data["smoothed TR"]) * 100

    data["DX"] = (100 * abs(data["+DI"] - data["-DI"])) / (abs(data["+DI"]) + abs(data["-DI"]))

    # 5. Average Directional Index
    adx_values = {}
    initial = data["DX"].iloc[period:2 * period].mean()

    adx_values[2 * period - 1] = initial

    for i in range(2 * period, len(data)):
        adx_values[i] = ((period - 1) * adx_values[i - 1] + data["DX"].iloc[i]) / period

    data["adx"] = pd.Series(list(adx_values.values()), index=data.index[2 * period - 1:])

    data = data[["adx", "DX", "+DI", "-DI", "smoothed TR"]]

    return data

In [6]:
def rolling_price_std(data, period):

    rolling_std = data["Close_CAD"].pct_change(fill_method=None).rolling(period).std()

    return rolling_std

In [7]:
def pull_beta(ticker):

    info = ticker.info
    beta = info.get("beta")

    return beta

In [8]:
ADX_PERIODS = {"1M": 21, "3M": 63, "6M": 126}

def indicator_application(ticker, ticker_df, start_date):
    ticker = yf.Ticker(ticker)

    ticker_df["Market Cap"] = pull_market_cap(ticker, ticker_df,start_date)

    ticker_df["Rolling STD 63d"] = rolling_price_std(ticker_df, 63)

    ticker_df["Beta"] = pull_beta(ticker)

    for label, period in ADX_PERIODS.items():
        adx_df = calculate_adx(ticker_df, period=period)
        adx_df = adx_df.add_suffix(f" {label}")
        
        adx_df[f"DI spread {label}"] = (adx_df[f"+DI {label}"] - adx_df[f"-DI {label}"])
        ticker_df = ticker_df.drop(columns=adx_df.columns, errors="ignore")
        ticker_df = pd.concat([ticker_df, adx_df], axis=1)

    return ticker_df

In [9]:
# Pulls the return on equity for a stock
def pull_roe(ticker_symbol):
    ticker = yf.Ticker(ticker_symbol)

    info = ticker.info
    roe = info.get("returnOnEquity")

    if roe != None:
        roe = roe * 100

    return roe


In [10]:
# Pulls trailing and forward EPS for a stock
def pull_eps(ticker_symbol):
    ticker = yf.Ticker(ticker_symbol)

    info = ticker.info

    trailing_eps = info.get("trailingEps")
    forward_eps = info.get("forwardEps")

    return trailing_eps, forward_eps




In [11]:
# Calculates the correlation between stocks
def calculate_correlations(prices):
    returns = prices.pct_change(fill_method=None)
    return returns.corr()


# Calculates the average correlation with selected stocks
def average_correlation(candidate, selected_stocks, correlations):
    return correlations.loc[candidate, selected_stocks].mean()



In [12]:
for ticker, ticker_data in stock_data.items():
    stock_data[ticker] = indicator_application(ticker, ticker_data, end_dl)

In [13]:
# Calculates the total portfolio weight in each sector
def calculate_sector_weights(portfolio):
    sector_weights = portfolio.groupby("Sector")["Weight"].sum()
    return sector_weights


# Checks if any sector exceeds the 40% limit we have
def check_sector_limit(portfolio):
    sector_weights = calculate_sector_weights(portfolio)
    return (sector_weights <= 0.40 + 0.0000000001).all()




## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

In [14]:
# basically the length of our competition 
HOLD_DAYS = 6 

series_by_ticker = {}
for ticker, ticker_data in stock_data.items():
  series_by_ticker[ticker] = ticker_data["Close_CAD"]

close = pd.DataFrame(series_by_ticker)

returns = close.pct_change(fill_method=None)

fwd_returns = close.shift(-HOLD_DAYS) / close - 1

print(f"Close Table Shape: {close.shape}")

Close Table Shape: (999, 35)


In [15]:
HIGH_RET_RULES = {
    #"adx 6M": ">",
    #"DI spread 6M": ">",
    "adx 3M": ">",
    "DI spread 3M": ">",
    "adx 1M": ">",
    "DI spread 1M": ">",
#    "Beta": ">",
 #   "Market Cap": "<",
}

In [16]:
def build_tables(rules):

    #essentially evert indicator in rules gets its own column 

    tables = {} 
    for col in rules: 
        col_data = {}
        for t, df in stock_data.items():
            col_data[t] = df[col]
        tables[col] = pd.DataFrame(col_data)
    return tables

high_return_tables = build_tables(HIGH_RET_RULES)

In [17]:
first_table = high_return_tables["adx 3M"]

has_indicators = first_table.notna().any(axis=1)
has_future_return = fwd_returns.notna().any(axis=1)

has_enough_prices = close.notna().sum(axis=1) >= 10

usable = first_table.index[
    has_indicators & has_future_return & has_enough_prices
]
test_dates = usable[::-HOLD_DAYS][:64][::-1]

split = int(len(test_dates) * 0.7)
tune_dates = test_dates[:split]
validation_dates = test_dates[split:]

print(len(tune_dates), "tuning weeks:    ", tune_dates[0], "->", tune_dates[-1])
print(len(validation_dates), "validation weeks:", validation_dates[0], "->", validation_dates[-1])

44 tuning weeks:     2025-05-16 -> 2026-05-28
20 validation weeks: 2026-06-05 -> 2026-11-17


In [18]:
def passes(tables, rules, cutoffs, date):
    mask = pd.Series(True, index=close.columns)

    for (col, direction), cutoff in zip(rules.items(), cutoffs):
        values = tables[col].loc[date]

        if direction == ">":
            this_rule = values > cutoff
        else:
            this_rule = values < cutoff

        mask = mask & this_rule 
    
    return mask 


def make_space(tables, rules, dates):
    space = []
    for col, direction in rules.items():

        if direction == ">":
            q = [0.05, 0.90] 
        else:
            q = [0.10, 0.95]

        lo, hi = tables[col].loc[dates].stack().quantile(q)

        space.append(Real(float(lo), float(hi), name=col))

    return space


hr_space = make_space(high_return_tables, HIGH_RET_RULES, tune_dates)

In [19]:
# Calculate a growth score using ADX and DI spread
def growth_scores(picks, date):
    scores = {}

    for ticker in picks:
        adx = high_return_tables["adx 3M"].loc[date, ticker]
        di = high_return_tables["DI spread 3M"].loc[date, ticker]

        # Only reward positive directional movement
        if pd.isna(adx) or pd.isna(di) or di <= 0:
            score = 0
        else:
            score = adx * di

        scores[ticker] = score

    return pd.Series(scores)


In [20]:

def growth_weights(scores):
    # Give every stock an equal base weight
    equal_weights = pd.Series(1 / len(scores), index=scores.index)

    # Give stronger stocks a larger weight
    if scores.sum() > 0:
        score_weights = scores / scores.sum()
    else:
        score_weights = equal_weights

    # Combine equal weighting and growth-based weighting
    weights = 0.5 * equal_weights + 0.5 * score_weights

    return weights



In [21]:

def cap_stock_weights(weights):
    weights = weights.copy()

    # Reduce any stock above 15%
    while weights.max() > 0.15:
        excess = 0

        for ticker in weights.index:
            if weights[ticker] > 0.15:
                excess += weights[ticker] - 0.15
                weights[ticker] = 0.15

        # Give the extra weight to stocks below the limit
        available = weights[weights < 0.15]

        if len(available) == 0:
            break

        weights.loc[available.index] += excess * available / available.sum()

    return weights


# Save the sector for each eligible stock
stock_sectors = {}

for ticker in stock_data:
    stock_sectors[ticker] = yf.Ticker(ticker).info.get("sector")
    

In [22]:
def cap_sector_weights(weights, stock_sectors):
    weights = weights.copy()

    for _ in range(100):
        sector_totals = {}

        for ticker in weights.index:
            sector = stock_sectors[ticker]
            sector_totals[sector] = sector_totals.get(sector, 0) + weights[ticker]

        # Find sectors above 40%
        over = [s for s in sector_totals if sector_totals[s] > 0.40 + 1e-9]

        if not over:
            return weights

        sector = over[0]
        excess = sector_totals[sector] - 0.40

        # Reduce weights within the overweight sector
        for ticker in weights.index:
            if stock_sectors[ticker] == sector:
                weights[ticker] *= 0.40 / sector_totals[sector]

        # Find stocks in other sectors with room for more weight
        available = [
            t for t in weights.index
            if stock_sectors[t] != sector
            and weights[t] < 0.15
            and sector_totals[stock_sectors[t]] < 0.40
        ]

        if not available:
            raise ValueError("Not enough room to redistribute sector weights")

        # Redistribute excess gradually without exceeding either limit
        while excess > 1e-9:
            room = {
                t: min(0.15 - weights[t],
                       0.40 - sector_totals[stock_sectors[t]])
                for t in available
            }
            room = {t: r for t, r in room.items() if r > 1e-9}

            if not room:
                raise ValueError("Not enough room to redistribute sector weights")

            ticker = max(room, key=room.get)
            amount = min(excess, room[ticker])
            weights[ticker] += amount
            sector_totals[stock_sectors[ticker]] += amount
            excess -= amount

    raise ValueError("Could not satisfy sector limits")


In [23]:
def final_growth_weights(picks, date):
    scores = growth_scores(picks, date)
    weights = growth_weights(scores)

    weights = cap_stock_weights(weights)
    weights = cap_sector_weights(weights, stock_sectors)

    return weights



In [24]:
def select_growth_stocks(date, passed):
    adx = high_return_tables["adx 3M"].loc[date]

    # Rank stocks that passed filters first
    passed_stocks = list(adx[passed].dropna().sort_values(ascending=False).index)

    # Remaining stocks are backups
    backup_stocks = list(adx[~passed].dropna().sort_values(ascending=False).index)

    selected = passed_stocks[:20]

    # Add backup stocks if fewer than 10 were selected
    if len(selected) < 10:
        for ticker in backup_stocks:
            if len(selected) >= 10:
                break

            selected.append(ticker)

    return selected


In [25]:
MIN_TICKERS = 10   #setting the punishment for min tickers above the min of 10. I'd rather have more than less. 
MAX_TICKERS = 20   # max 


def weekly_high_return(cutoffs, dates):
    """For each date: pick tickers, hold 6 days, record the return. NaN = too few passed."""
    out = []

    for i in dates:
        passed = passes(high_return_tables, HIGH_RET_RULES, cutoffs, i)

        picks = select_growth_stocks(i, passed)

        if len(picks) < MIN_TICKERS:
            out.append(np.nan)
            continue

    
        try:
             weights = final_growth_weights(picks, i)
        except ValueError:
            out.append(np.nan)
            continue
            
        weekly_return = (fwd_returns.loc[i, picks] * weights).sum()
        out.append(weekly_return)

    return pd.Series(out, index=dates)


def count_passing(tables, rules, cutoffs, dates):
    """How many tickers passed the rules on each date."""
    counts = []
    for i in dates:
        counts.append(passes(tables, rules, cutoffs, i).sum())
    return pd.Series(counts, index=dates)


def score_high_return(cutoffs, dates):
    """ONE number for these cutoffs: average weekly return, with penalties for failed weeks."""
    weekly = weekly_high_return(cutoffs, dates)
    n = count_passing(high_return_tables, HIGH_RET_RULES, cutoffs, dates)

    shortfall = (MIN_TICKERS - n).clip(lower=0)

    weekly = weekly.fillna(-0.10)

    weekly = weekly - 0.01 * shortfall

    return weekly.mean()

In [26]:
result_high_returns = gp_minimize(

    lambda p: -score_high_return(p, tune_dates),
    hr_space, 
    n_calls=30,
    n_initial_points=10,
    acq_func="EI", 
    random_state=0,
    verbose=True
)

best_hr = result_high_returns.x 

# no cutoff comparison 
no_cut_hr = [-np.inf if d == ">" else np.inf for d in HIGH_RET_RULES.values()]


print("best cutoffs:", dict(zip(HIGH_RET_RULES, [round(float(x), 2) for x in best_hr])))
print()


print("                tune      val")
print("tuned:      ", round(score_high_return(best_hr, tune_dates), 4),
      "  ", round(score_high_return(best_hr, validation_dates), 4))
print("no cutoffs: ", round(score_high_return(no_cut_hr, tune_dates), 4),
      "  ", round(score_high_return(no_cut_hr, validation_dates), 4))
print()


tuned_weeks = weekly_high_return(best_hr, validation_dates)
base_weeks = weekly_high_return(no_cut_hr, validation_dates)

wins = (tuned_weeks > base_weeks).sum()
print(wins, "wins out of", len(validation_dates), "validation weeks")


print("tickers passing each validation week:",
      count_passing(high_return_tables, HIGH_RET_RULES, best_hr, validation_dates).tolist())

Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.1217
Function value obtained: 0.0827
Current minimum: 0.0827
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.1201
Function value obtained: 0.0485
Current minimum: 0.0485
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.1146
Function value obtained: 0.0564
Current minimum: 0.0485
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.1166
Function value obtained: 0.0546
Current minimum: 0.0485
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.1149
Function value obtained: 0.0689
Current minimum: 0.0485
Iteration No: 6 started. Evaluating

In [27]:

# Test the growth scores on the most recent data
latest = high_return_tables["adx 3M"].dropna(how="all").index[-1]

passed = passes(high_return_tables, HIGH_RET_RULES, best_hr, latest)

growth_picks = select_growth_stocks(latest, passed)

scores = growth_scores(growth_picks, latest)

print(scores.sort_values(ascending=False))


BIIB    277.100615
ABBV    251.589651
KO      196.582754
LLY     192.502426
PFE     145.953570
T.TO    131.199188
ACN     125.700128
CL      111.063513
MRK     104.185209
UNP      77.633784
MO       72.025486
AAPL     54.602309
UPS      54.515173
LMT      39.070457
BAC       0.000000
dtype: float64


In [28]:
def show_passing(cutoffs, dates, show_picks=True):
    for i in dates:
        passed = passes(high_return_tables, HIGH_RET_RULES, cutoffs, i)
        names = list(passed[passed].index)
        print(f"{i}  ({len(names)} passed): {names}")

        if show_picks and len(names) >= MIN_TICKERS:
            picks = select_growth_stocks(i, passed)
            dropped = [t for t in names if t not in picks]
            print(f"     held ({len(picks)}): {list(picks)}")
            if dropped:
                print(f"     passed but cut by top-{MAX_TICKERS}: {dropped}")
        print()
latest = high_return_tables["adx 3M"].dropna(how="all").index[-1]
show_passing(best_hr, [latest])

2026-11-25  (15 passed): ['AAPL', 'ABBV', 'ACN', 'BAC', 'BIIB', 'CL', 'KO', 'LLY', 'LMT', 'MO', 'MRK', 'PFE', 'T.TO', 'UNP', 'UPS']
     held (15): ['T.TO', 'BIIB', 'KO', 'ABBV', 'MRK', 'LLY', 'PFE', 'ACN', 'BAC', 'UNP', 'CL', 'AAPL', 'UPS', 'MO', 'LMT']



In [29]:

# Calculate weights for the newly selected stocks
final_weights = final_growth_weights(growth_picks, latest)

# Check stock weights
print("Number of stocks:", len(final_weights))
print("Total weight:", round(final_weights.sum() * 100, 2), "%")
print("Largest stock:", round(final_weights.max() * 100, 2), "%")

# Check sector weights
growth_portfolio = pd.DataFrame({
    "Ticker": final_weights.index,
    "Sector": [stock_sectors[t] for t in final_weights.index],
    "Weight": final_weights.values
})

print()
print(calculate_sector_weights(growth_portfolio))
print("Sector limits passed:", check_sector_limit(growth_portfolio))


Number of stocks: 15
Total weight: 100.0 %
Largest stock: 10.09 %

Sector
Communication Services    0.069107
Consumer Defensive        0.203525
Financial Services        0.064852
Healthcare                0.400000
Industrials               0.146686
Technology                0.115830
Name: Weight, dtype: float64
Sector limits passed: True


### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

For our growth portfolio that is focused on high returns, we focused on selecting stocks that show strong upward price trends. We used ADX to measure the strength of a stock's trend and DI spread to determine whether upward or downward price movement is stronger. In addition, we also considered beta and market capitilization when selecting stocks. To help determine which stock to select, we used Gaussian Process Optimization. This method tests different thresholds for our indicators using historical stock returns. Based on the results of previous tests, it chooses new combinations of thresholds to evaluate, helping us find settings that performed relatively well without testing every possible combination.

In terms of the weighting, we decided to use a combination of 50% equal weighting and 50% growth-based weighting. We started by distributing half of the portfolio equally among all selected stocks, ensuring that every stock received a base allocation. We then distributed the remaining half based on each stock's growth score, which we calculated by multiplying its ADX by its positive DI spread. Stocks with stronger positive trends received a larger portion of this remaining allocation.

The reason we chose this approach is because we wanted to prioritize stocks showing stronger growth signals without fully relying on those signals. Using equal weighting for half of the portfolio helps maintain diversification and reduce risk of putting too much into only a few stocks. Simultaneously, the growth based portion allows us to take on additional risk in pursuit of a higher return, fitting client A's objective.

Lastly, we capped each individual stock at 15% of the portfolio and limited each sector to a maximum of 40% of the total portfolio value to follow the assignment's requirements. This also helps reduce concentration risk and maintain diversification.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

We compared our portfolio to an equal weighted portfolio holding the exact same stocks, from October 1, 2025 to the most recent date in our data. When we ran it on November 26, 2026, our portfolio returned 30.79% compared to 28.95% for equal weighting. This means our weighting added about 1.84% more than the equal weighted portfolio. Giving more money to stocks with stronger growth scores helped the portfolio earn more than just splitting the money evenly. Our daily volatility was slightly higher, 0.967% compared to 0.929% for the equal weighted portfolio, which makes sense considering Client A wanted the highest returns and was willing to accept more risk.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

One important limitation of our growth strategy for client A is that it relies heavily on historical price trends. Although ADX and DI spread can help us identify stocks showing strong upward momentum, this does not guarantee that these trends will continue. 

For example, unexpected earnings reports, change in interest rates, or economic downturns could cause these stocks to suddenly drop in value. Since we give higher weights to stocks with higher growther scores, a sudden reversal in these stocks could have a larger negative impact on our portfolio's overall return. Consequently, tthis can increase the portfolio's standard deviation, translating into greater risk and volatility.

Another limitation is the Gaussian Process. Since this process uses historical data to find indicator thresholds. These thresholds might perform well on past data but may not work as effectively under different market conditions. Although we use diversification and sector limits to manage risk, they cannot completely protect our portfolio from market wide losses.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

In [30]:
# CONSTANTS
lv_window = 63
MAX_TICKERS = 20 #same variables just restating for clarity
MIN_TICKERS = 10 #same variables just restating for clarity
sector_cap = 0.4
max_per_sector = 8  

min_tickers = MIN_TICKERS  # made minimum 15 because portfolio volatility between 10 and 15 tickers are more or less the same. 
# Having it at 15 for sanity purposes lol .

#More testing determins that the volatility between 10 and 20 tickers by the strategy are more or less the same. 15 is in the sweet spot
# in the middle so yeah we go with 15. :)


sector_list = [
    "Utilities",
    "Consumer Defensive",
    "Healthcare",
    "Real Estate",
    "Financial Services",
    "Industrials",
    "Energy",
    "Basic Materials",
    "Communication Services",
    "Consumer Cyclical",
    "Technology"
]

In [31]:
def pull_stock_sectors (tickers):
    sectors = []

    for ticker in tickers:
        stock = yf.Ticker(ticker)
        sector = stock.info.get("sector")

        sectors.append({
            "Ticker": ticker,
            "Sector" :sector
        })
    return pd.DataFrame(sectors)


sectors = pull_stock_sectors(stock_data.keys())
print(sectors)

     Ticker                  Sector
0      AAPL              Technology
1      ABBV              Healthcare
2       ABT              Healthcare
3       ACN              Technology
4       AIG      Financial Services
5      AMZN       Consumer Cyclical
6       AXP      Financial Services
7        BA             Industrials
8       BAC      Financial Services
9     BB.TO              Technology
10     BIIB              Healthcare
11      BLK      Financial Services
12      BMY              Healthcare
13        C      Financial Services
14      CAT             Industrials
15       CL      Consumer Defensive
16       KO      Consumer Defensive
17      LLY              Healthcare
18      LMT             Industrials
19       MO      Consumer Defensive
20      MRK              Healthcare
21      PEP      Consumer Defensive
22      PFE              Healthcare
23    RY.TO      Financial Services
24       PM      Consumer Defensive
25     PYPL      Financial Services
26     QCOM              Tec

In [32]:
def lookback_returns(date, window=lv_window):
    window_returns = returns.loc[:date].tail(window)
    return window_returns

def portfolio_vol(window_returns, tickers):
    port_returns = window_returns[tickers].mean(axis=1)
    return port_returns.std()

sector_of = sectors.set_index("Ticker")["Sector"].to_dict()

def get_sector(ticker):
    s = sector_of.get(ticker)
    if s is None:
        return (f"Unknown_{ticker}")
    return s


def sector_counts(tickers):
    counts = {}
    for t in tickers:
        s = get_sector(t)
        counts[s] = counts.get(s, 0) + 1
    return counts

def over_cap_sectors(selected):
    if len(selected) == 0:
        return set()
    over = set()
    for s, c in sector_counts(selected).items():
        if c / len(selected) > sector_cap + 1e-9:
            over.add(s)
    return over

def usable_universe(window_returns, min_coverage=0.9):
    coverage = window_returns.notna().mean()
    return list(coverage[coverage >= min_coverage].index)

def eligible_candidates(selected, universe):
    counts = sector_counts(selected)
    over = over_cap_sectors(selected)

    base = []
    for t in universe:
        if t not in selected and counts.get(get_sector(t), 0) < max_per_sector:
            base.append(t)

    preferred = [t for t in base if get_sector(t) not in over]

    if len(preferred) > 0:
        return preferred
    return base 

In [33]:
def build_low_vol(date, universe=None, verbose=False, min_n= min_tickers, max_n=MAX_TICKERS):
    window = lookback_returns(date)

    if universe is None:
        universe = usable_universe(window)

    selected = []
    current_vol = np.inf 

    while len(selected) < max_n:
        candidates = eligible_candidates(selected, universe)
        if len(candidates) == 0:
            break

        best_ticker = None
        best_vol = np.inf
        for i in candidates:
            v = portfolio_vol(window, selected + [i])
            if pd.notna(v) and v < best_vol:
                best_vol = v
                best_ticker = i

        if best_ticker is None:
            break

        # STOPPING RULE: all three must be true to stop. subject to change depending on tests(don't forget leon)
        raises_vol = best_vol >= current_vol
        sectors_ok = len(selected) > 0 and len(over_cap_sectors(selected)) == 0
        big_enough = len(selected) >= min_n

        if raises_vol and sectors_ok and big_enough:
            if verbose:
                print(f"STOP: adding {best_ticker} would raise vol to {best_vol:.5f}")
            break

        selected.append(best_ticker)
        current_vol = best_vol

        if verbose:
            print(f"N={len(selected):2d}  added {best_ticker:8s} ({get_sector(best_ticker)})  vol={current_vol:.5f}")

    # warnings 
    if len(over_cap_sectors(selected)) > 0:
        print(f"WARNING {date}: sectors over cap: {over_cap_sectors(selected)}")
    if len(selected) < min_n:
        print(f"WARNING {date}: only {len(selected)} tickers selected (min {min_n})")

    return selected

In [34]:
def min_var_weights(tickers, window_returns, lo=0.05, hi=0.15, sector_max=0.40):
    R = window_returns[tickers].dropna()
    cov = R.cov().values
    n = len(tickers)

    # weighting must equal 1 :)
    cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]

    # Sector condition
    for s in set(get_sector(t) for t in tickers):
        idx = [k for k, t in enumerate(tickers) if get_sector(t) == s]
        cons.append({"type": "ineq", "fun": lambda w, idx=idx: sector_max - w[idx].sum()})

    # Minimizing variance 
    res = minimize(lambda w: w @ cov @ w, np.ones(n) / n,
                   bounds=[(lo, hi)] * n, constraints=cons, method="SLSQP")

    if not res.success:
        raise ValueError("Min-variance optimization failed: " + res.message)

    return pd.Series(res.x, index=tickers)

In [35]:

def make_low_vol_portfolio(selected, date):

    n = len(selected)
    if n == 0:
        print("ERROR: empty portfolio")
        return pd.DataFrame(columns=["Ticker", "Sector", "Weight"])

    weights = min_var_weights(selected, lookback_returns(date))

    portfolio = pd.DataFrame({
        "Ticker": selected,
        "Sector": [get_sector(t) for t in selected],
        "Weight": weights.values,
    })

    problems = []
    if not (min_tickers <= n <= MAX_TICKERS):
        problems.append(f"N={n} outside [{min_tickers}, {MAX_TICKERS}]")
    if abs(portfolio["Weight"].sum() - 1) > 1e-9:
        problems.append(f"weights sum to {portfolio['Weight'].sum()}")
    if portfolio["Weight"].min() < 0.05 - 1e-9 or portfolio["Weight"].max() > 0.15 + 1e-9:
        problems.append("a weight is outside 5%-15%")
    if not check_sector_limit(portfolio):
        problems.append(f"sector over 40%: {calculate_sector_weights(portfolio).to_dict()}")

    if problems:
        print("PORTFOLIO CHECK FAILED:")
        for p in problems:
            print("   -", p)
    else:
        print(f"All checks passed ({n} tickers, weights {portfolio['Weight'].min():.2%} to {portfolio['Weight'].max():.2%})")

    return portfolio

In [36]:
latest = returns.dropna(how="all").index[-1]
lv_picks = build_low_vol(latest, verbose=True)
lv_portfolio = make_low_vol_portfolio(lv_picks, latest)
print(lv_portfolio)
print(calculate_sector_weights(lv_portfolio))

N= 1  added RY.TO    (Financial Services)  vol=0.01151
N= 2  added KO       (Consumer Defensive)  vol=0.00870
N= 3  added PFE      (Healthcare)  vol=0.00880
N= 4  added TD.TO    (Financial Services)  vol=0.00881
N= 5  added MO       (Consumer Defensive)  vol=0.00861
N= 6  added AMZN     (Consumer Cyclical)  vol=0.00830
N= 7  added T.TO     (Communication Services)  vol=0.00749
N= 8  added LMT      (Industrials)  vol=0.00725
N= 9  added CAT      (Industrials)  vol=0.00723
N=10  added PM       (Consumer Defensive)  vol=0.00718
STOP: adding AIG would raise vol to 0.00722
All checks passed (10 tickers, weights 10.00% to 10.00%)
  Ticker                  Sector  Weight
0  RY.TO      Financial Services     0.1
1     KO      Consumer Defensive     0.1
2    PFE              Healthcare     0.1
3  TD.TO      Financial Services     0.1
4     MO      Consumer Defensive     0.1
5   AMZN       Consumer Cyclical     0.1
6   T.TO  Communication Services     0.1
7    LMT             Industrials     0.1

In [37]:
window = lookback_returns(latest)
w = lv_portfolio.set_index("Ticker")["Weight"]
mv_vol = (window[w.index] * w).sum(axis=1).std()
eq_vol = window[w.index].mean(axis=1).std()
print(f"Min-variance daily vol: {mv_vol:.5f}   Equal-weight daily vol: {eq_vol:.5f}")

Min-variance daily vol: 0.00713   Equal-weight daily vol: 0.00718


### Client B — Financial justification

**Why does this portfolio make financial sense?** 

Since client B wants the lowest risk possible, our main goal was to build a portfolio that fluctuates as little as possible. We measured risk using the standard deviation of daily returns over the last 63 trading days.

Instead of just picking the stocks that were the least volatile on their own, we added stocks one at a time. Each time, we chose the stock that lowered the portfolio’s overall volatility. We did it this way because if one stock tends to go up while another goes down, holding both can make the portfolio steadier as a whole. We kept adding stocks until another one would make the portfolio more volatile, while keeping between 10 and 20 stocks.

We gave each stock the same weight so no one stock could affect the portfolio too much. We also kept each sector under 40% and each stock between 5% and 15% to diversify our portfolio.


**What evidence supports your strategy?** We compared our portfolio to an equal weighted portfolio holding the exact same stocks, from October 1, 2025 to the most recent date in our data. Since our portfolio already gives each stock the same weight, both had identical results when we ran it on November 26, 2026 We had a 19.25% return and 0.73% daily volatility. This shows our low risk comes from the stocks we selected, not the weights. A daily volatility of 0.73% is low,and much lower than Client A's 0.967%, which supports our goal of minimizing risk. 

**What could go wrong?** 

The biggest limitation is that we are using the past three months data to guess how the stocks will behave in the future. Just because a stock was stable recently does not mean it will stay that way. Something like a company reporting lower revenue or gross margin than expected, or cuts its dividend could make its stock price much more volatile. Since we measure everything in CAD, a sudden change in the USD/CAD excahnge rate could make our US stocks more volatile.

Another issue is that during a big market crash, most stocks tend to drop together. In that case, the benefit we got from picking stocks that move differently could disappear. Also, since we focused on keeping risk low, our portfolio probably won't keep up with riskier stocks when the market is doing really well. 

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

In [38]:
start_date = "2025-10-01"    #comparison period: oct 1,2025 to latest date

def portfolio_value(tickers, weights):

    prices = pd.DataFrame()
    for i in range(len(tickers)):
        prices[tickers[i]] = stock_data[tickers[i]]["Close_CAD"]

    prices = prices.ffill()                  
    prices = prices.loc[start_date:]         
    prices = prices.dropna()

    value = 0
    for i in range(len(tickers)):
        shares = weights[i] / prices[tickers[i]].iloc[0]
        value = value + shares * prices[tickers[i]]

    return value

def compare(tickers, our_weight):

    equal_weight = []
    for i in range(len(tickers)):
        equal_weight.append(1 / len(tickers))

    ours = portfolio_value(tickers, our_weight)
    equal = portfolio_value(tickers, equal_weight)

    table = pd.DataFrame()
    table["Portfolio"] = ["Our weights", "Equal weights"]
    table["Return (%)"] = [(ours.iloc[-1] / ours.iloc[0] - 1) * 100,
                           (equal.iloc[-1] / equal.iloc[0] - 1) * 100]
    table["Daily volatility (%)"] = [ours.pct_change().std() * 100,
                                     equal.pct_change().std() * 100]

    print("Period:", ours.index[0], "->", ours.index[-1])

    return table.round(5)

#client A: tickers and weights from the growth strategy
tickers_A = list(final_weights.index)
weights_A = list(final_weights.values)

#client B: tickers and weights from the low volatility strategy
tickers_B = list(lv_portfolio["Ticker"])
weights_B = list(lv_portfolio["Weight"])

print("Client A")
display(compare(tickers_A, weights_A))

print("Client B")
display(compare(tickers_B, weights_B))

Client A
Period: 2025-10-01 -> 2026-11-25


,Portfolio,Return (%),Daily volatility (%)
0,Our weights,30.79124,0.96663
1,Equal weights,28.95294,0.92851


Client B
Period: 2025-10-01 -> 2026-11-25


,Portfolio,Return (%),Daily volatility (%)
0,Our weights,19.25019,0.73187
1,Equal weights,19.25019,0.73187


## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

In [39]:

# Client A - Growth Portfolio
budget_A = 1000000

# Final date for the portfolio
#target_date = "2026-11-27"

# For testing purposes as of Nov 25th, change the line up top to the commented line below
target_date = "2026-11-25"

# Find the latest available trading date on or before our target
available_dates = high_return_tables["adx 3M"].dropna(how="all").index
latest = available_dates[available_dates <= target_date][-1]

# Select our growth stocks
passed = passes(high_return_tables, HIGH_RET_RULES, best_hr, latest)
growth_picks = select_growth_stocks(latest, passed)

# Calculate the target weights
final_weights = final_growth_weights(growth_picks, latest)

print("Target date:", target_date)
print("Date used:", latest)
print("Number of stocks:", len(growth_picks))
print("Total weight:", round(final_weights.sum() * 100, 2), "%")
print()
print((final_weights * 100).round(2))


Target date: 2026-11-25
Date used: 2026-11-25
Number of stocks: 15
Total weight: 100.0 %

T.TO     6.91
BIIB    10.09
KO       8.69
ABBV     9.45
MRK      5.72
LLY      7.96
PFE      6.78
ACN      6.76
BAC      6.49
UNP      5.45
CL       6.36
AAPL     4.82
UPS      4.82
MO       5.30
LMT      4.40
dtype: float64


### Contribution and AI Declaration ###
**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.